In [ ]:

import os
import sys
import time
import subprocess
import threading
import keyboard

# CONFIGURATION
PROGRAM_A = r"C:\Path\To\ProgramA.py"
PROGRAM_B = r"C:\Path\To\ProgramB.py"
TRIGGER_KEY = "f8"
INTERVAL = 180

PYTHON = sys.executable

a_lock = threading.Lock()
b_lock = threading.Lock()


def run_program(path, lock):
    if not os.path.isfile(path):
        print(f"File not found: {path}")
        return

    if not lock.acquire(blocking=False):
        return

    try:
        subprocess.run(
            [PYTHON, path],
            cwd=os.path.dirname(path) or None,
            check=False
        )
    except Exception as error:
        print(f"Error running {path}: {error}")
    finally:
        lock.release()


def on_key_press(event):
    if event.name == TRIGGER_KEY and event.event_type == "down":
        if not b_lock.locked():
            threading.Thread(
                target=run_program,
                args=(PROGRAM_B, b_lock),
                daemon=True
            ).start()


keyboard.hook(on_key_press)

print("Controller started.")
print(f"Program A: every {INTERVAL} seconds")
print(f"Program B: when {TRIGGER_KEY.upper()} is pressed")

next_run = time.monotonic()

try:
    while True:
        if time.monotonic() >= next_run:
            threading.Thread(
                target=run_program,
                args=(PROGRAM_A, a_lock),
                daemon=True
            ).start()

            next_run = time.monotonic() + INTERVAL

        time.sleep(0.1)

except KeyboardInterrupt:
    keyboard.unhook_all()
    print("Controller stopped.")



Controller started.
Program A: every 180 seconds
Program B: when F8 is pressed
File not found: C:\Path\To\ProgramA.py
File not found: C:\Path\To\ProgramB.py
File not found: C:\Path\To\ProgramB.py
File not found: C:\Path\To\ProgramA.py
File not found: C:\Path\To\ProgramA.py
